In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_iva_condition_comparison import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 06 — IVA decomposition across conditions (joined separate)

Runs the IVA decomposition that pools Placebo and Psilocybin on the **subject** axis: each recording is its own dataset, so a participant occupies two rows with two mixing matrices and the topographies are per recording.\n\nAdd `--store_components` to write the result for [`iva_component_analysis_joined_separate.ipynb`](iva_component_analysis_joined_separate.ipynb) to analyse.

**The method is documented in
[`docs/analyses/06-iva-condition-comparison/`](../../docs/analyses/06-iva-condition-comparison/README.md)** — this
notebook does not restate it.

**What this notebook is for.** Choosing parameters and checking the workflow cheaply
before committing a node to it. It calls `run()` from
[`scripts/run_iva_condition_comparison.py`](../../scripts/run_iva_condition_comparison.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# The decomposition to run. Small on purpose -- the full-size run is the job's.
EXPERIMENT = "assr"
N_PCA = 5
# None = broadband. (30, 50) = the gamma-band run of the *_gamma.pbs jobs: the
# frequency axis is cut BEFORE the fit, and figures land under bands/ as 30-50hz_*.
BAND_RANGE = None

# Subset every axis that can be subset. Channels and pairs are what make this cheap;
# the wavelet transform itself is reused from the cache rather than recomputed.
N_PAIRS = 6  # participant pairs
N_CHANNELS = 32  # the ASSR mask needs --lenient_mask at this width
N_TIMES = 3000  # samples (~12 s at 250 Hz)

OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "out"
    / "joined_separate"
)
PLOTS_DIR = OUT_DIR / "plots"

## Run

The same function the PBS job calls. Keep the scale small here: the full-size pass is what
[`jobs/metacentrum/06-iva-condition-comparison/`](../../jobs/metacentrum/06-iva-condition-comparison) is for.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--n_pca",
        str(N_PCA),
        *(["--band_range", *map(str, BAND_RANGE)] if BAND_RANGE else []),
        "--n_pairs",
        str(N_PAIRS),
        "--n_channels",
        str(N_CHANNELS),
        "--n_times",
        str(N_TIMES),
        "--reuse_wavelets",
        "--lenient_mask",  # a 32-channel subset drops most ASSR electrodes
        "--save_dir",
        str(PLOTS_DIR),
    ]
)
run(args)

## Figures

Written by the shared plotters in
[`src/visualization/iva_condition_plots.py`](../../src/visualization/iva_condition_plots.py),
which the CLI uses too -- so what is shown here is what the job produces.

In [ ]:
for png in sorted(PLOTS_DIR.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))